# Dev-set classifier runner

Runs the locked `classifier_lib` prompt against the **50-chart development set** drawn from the 312-encounter validation cohort (the primary adjudicator's master file). Test calls Azure OpenAI per encounter under the institutional BAA, validates each response against `ChartClassification`, enforces a per-run cost ceiling, retries on transient errors, and writes outputs + run metadata as JSONL + JSON.

**Run on Minerva only.** This notebook touches real chart text. Outputs and the dev/eval split CSV live in gitignored directories. Run `probe_data.ipynb` first on any new data files to confirm column names and key cardinalities before this runner is trusted.

**Eval safety.** The deterministic split refuses to score any encounter not in the dev partition unless `UNLOCK_EVAL = True` is set explicitly. The 62 double-coded (secondary-adjudicator overlap) encounters are excluded from the dev-eligible pool via `DOUBLE_CODED_FILE`, so the full overlap lands in eval per the design. Do not change the split seed after first run.

**GPT-5 caveat.** GPT-5 is a reasoning-family model: `temperature` is forced to default, and the API uses `max_completion_tokens` not `max_tokens`. Run-to-run variance is expected; this v1.0 runner makes a single call per chart, per design decision.

## 0. STANDALONE build — bootstrap classifier_lib.py

This notebook was generated by `make_standalone.py` and embeds `classifier_lib.py` (sha256 `cc17d6284146302a…`). Run the next cell FIRST — it writes `classifier_lib.py` to the working directory so the import below succeeds.

**Do not edit the embedded library here.** Edit `classifier_lib.py` in the repo and regenerate. If this file and the repo disagree, the repo wins.

In [ ]:
%%writefile classifier_lib.py
"""Schema, prompt, and helpers for the police-transport ED chart classifier.

Single source of truth for:
- TransportMode and RuleFired enums
- ChartClassification Pydantic model
- PROMPT_VERSION, SYSTEM_PROMPT, USER_PROMPT_TEMPLATE
- format_notes helper

Both transport_classifier.ipynb (viewer) and run_classifier_dev.ipynb
(runner) import from this module. See docs/approach.md for the
operational definitions, decision-rule cascade, and change-control rules.
"""
from enum import Enum
from typing import Iterable, Literal

from pydantic import BaseModel, Field


PROMPT_VERSION = "v1.0"

PROMPT_VERSION_NOTES = """\
v1.0 (initial) — Codebook-derived definitions, NYC-ED cohort framing,
decision-rule cascade (R1–R8), rule_fired controlled vocabulary,
confidence rubric, structured-notes input format. Zero few-shot exemplars.

v1.0 pre-first-run amendment (2026-07-08, before any scored run existed,
so the version number is unchanged): added FDNY to the NYC abbreviation
list. FDNY EMS operates the NYC 911 ambulance system, so "BIB FDNY" is
EMS transport, not law enforcement. Motivated by the adjudicators' own
FDNY clarification during gold-standard coding.

v1.0 pre-first-run amendment #2 (2026-07-08): added the EXPLORATORY
custody_context field (+ custody_evidence_quote). No human gold standard
exists for custody — the adjudicators coded transport mode only — so
this field is reported descriptively with evidence-quote spot-checks and
is labeled as not human-validated in the manuscript. It answers the
custody-context question descriptively. transport_mode and
its codebook are unchanged.

v1.0 pre-first-run amendment #3 (2026-07-08): corrected the input
description. The real note pull contains ALL ED chart note types (ED
Psychiatric, ED Notes, ED Provider, Triage/Intake, nursing, consults,
progress notes, ...), not provider notes only — confirmed by the data
probe. The prompt now describes the actual input; the runner excludes
boilerplate types (Discharge Instructions, Attestation) before
classification. This also matches what the human adjudicators read.

v1.1 (2026-07-09) — TRIED AND REVERTED, NOT ADOPTED. Sharpened
police_and_ems to strict co-transport (mere police presence / bare EMS
mention excluded). Motivated by 50-chart dev error analysis. Result: it
made things WORSE on the same 50 (3-way kappa 0.519 -> 0.433; binary
police-vs-none 0.606 -> 0.532) because it pushed the model to credit
police LESS, whereas the primary adjudicator actually credits police
MORE (they code police-activated/accompanied EMS psych transports as
category 1, not 2 — contradicting the literal co-transport wording). The
1-vs-2 boundary proved rater-idiosyncratic and ambiguous, and at n=50 the
change was below the sampling-noise floor (a same-prompt v1.0 rerun swung
binary kappa 0.61->0.73 across two 50-chart draws). Reverted to v1.0 and
LOCKED for the held-out evaluation. v1.1 text is preserved in git history
(commit 72a62fe). See docs/approach.md change-control log.

Future versions are added during iteration on the 50-chart development
set. Each version must:
  - increment PROMPT_VERSION
  - log rationale + the specific disagreement(s) the change addresses
  - be locked before any of the 262-chart evaluation set is scored
  - have a corresponding entry in docs/approach.md change-control list
"""


class TransportMode(str, Enum):
    """Three-category transport classification based on documented
    transportation and custody at ED arrival.
    """

    ONLY_POLICE = "only_police"
    POLICE_AND_EMS = "police_and_ems"
    NO_POLICE = "no_police"


class CustodyContext(str, Enum):
    """EXPLORATORY secondary field: why police were involved at arrival.

    No human gold standard exists for this field (the adjudicators coded
    transport mode only), so it is reported descriptively and is not
    part of the PPV validation.
    """

    CRIMINAL_CUSTODY = "criminal_custody"
    INVOLUNTARY_PSYCH_HOLD = "involuntary_psych_hold"
    POLICE_INVOLVED_NO_CUSTODY = "police_involved_no_custody"
    NOT_APPLICABLE = "not_applicable"


# Numeric codes used by the human adjudicators in the gold-standard master
# file: 0 = not police transport, 1 = police transport, 2 = police + EMS.
# The validation notebook maps gold labels through this before comparing
# against classifier output.
GOLD_LABEL_MAP = {
    0: TransportMode.NO_POLICE,
    1: TransportMode.ONLY_POLICE,
    2: TransportMode.POLICE_AND_EMS,
}


class RuleFired(str, Enum):
    """Controlled vocabulary for which decision rule (R1–R8) produced the
    classification. One value per encounter.
    """

    TRANSPORT_BY_POLICE_ONLY = "transport_by_police_only"
    CO_TRANSPORT_POLICE_AND_EMS = "co_transport_police_and_ems"
    POLICE_PRESENT_AT_ARRIVAL_WITH_EMS = "police_present_at_arrival_with_ems"
    EMS_ONLY_TRANSPORT = "ems_only_transport"
    SELF_OR_OTHER_TRANSPORT = "self_or_other_transport"
    POLICE_INITIATED_EMS_ONLY = "police_initiated_ems_only"
    OFFICER_AS_PATIENT_NO_TRANSPORT = "officer_as_patient_no_transport"
    NO_LEO_MENTION = "no_leo_mention"
    AMBIGUOUS_DOCUMENTATION = "ambiguous_documentation"


class ChartClassification(BaseModel):
    """LLM output schema for a single ED encounter chart."""

    transport_mode: TransportMode = Field(
        ...,
        description=(
            "Transport-and-custody-at-arrival classification.\n\n"
            "- only_police: Patient brought to ED by law enforcement or in "
            "police custody, with no indication of EMS/ambulance transport.\n"
            "- police_and_ems: Both law enforcement and EMS involved in "
            "transport. Police must have actually transported or been "
            "physically present at arrival; merely initiating the EMS call "
            "does NOT qualify.\n"
            "- no_police: No evidence of LEO involvement in transport or "
            "custody. Includes EMS-only and self/other transport, and the "
            "officer-as-patient edge case when the officer was not "
            "transported or detained by police."
        ),
    )
    rule_fired: RuleFired = Field(
        ...,
        description=(
            "Controlled-vocabulary tag identifying which decision rule "
            "produced the transport_mode classification. See the "
            "decision-rules section of the system prompt for the rule "
            "cascade. Use AMBIGUOUS_DOCUMENTATION only when the notes are "
            "genuinely contradictory or silent on transport, and pair it "
            "with confidence='low'."
        ),
    )
    evidence_quote: str = Field(
        ...,
        description=(
            "Short verbatim quote from the notes that drove the "
            "classification (target <=200 chars). Empty string if no "
            "directly relevant text exists (rule_fired=no_leo_mention)."
        ),
    )
    confidence: Literal["high", "medium", "low"] = Field(
        ...,
        description=(
            "Self-rated confidence per the rubric in the system prompt:\n"
            "- high: arrival mode unambiguously documented.\n"
            "- medium: documented but with ambiguity (e.g., 'with police' — "
            "escort or just present?).\n"
            "- low: not documented, or contradictory across notes.\n"
            "Applies to transport_mode only, not custody_context."
        ),
    )
    custody_context: CustodyContext = Field(
        ...,
        description=(
            "EXPLORATORY: why police were involved at arrival.\n\n"
            "- criminal_custody: arrest-related custody (under arrest, "
            "handcuffed, prisoner, medical clearance for booking or "
            "arraignment, precinct/corrections custody).\n"
            "- involuntary_psych_hold: involuntary psychiatric removal or "
            "hold (EDP transport, 70/10, MHL 9.41 removal, involuntary "
            "evaluation) — NOT criminal custody even when officers use "
            "custody language.\n"
            "- police_involved_no_custody: police involved in transport or "
            "present at arrival but no custody or hold documented (e.g., "
            "assault victim driven in by officers).\n"
            "- not_applicable: required when transport_mode = no_police."
        ),
    )
    custody_evidence_quote: str = Field(
        ...,
        description=(
            "Short verbatim quote supporting custody_context (target <=200 "
            "chars). Empty string when custody_context is not_applicable or "
            "no directly relevant text exists."
        ),
    )


SYSTEM_PROMPT = """You are a clinical chart abstractor classifying how an emergency department (ED) patient arrived, based on the free-text ED provider notes from a single encounter.

# Setting and data

- Encounters are from a New York City emergency department.
- Charts have been pre-flagged in the electronic health record as potentially involving law enforcement; some flags are false positives, which is the point of this classification task.
- Input is the ED chart notes from a single encounter, presented in chronological order in the user message: provider notes (resident, attending, PA, NP), psychiatric notes, triage/intake and nursing documentation, consults, and progress/event notes. Arrival-mode language most often appears in triage/intake and nursing documentation and in the HPI of provider notes.
- Expect informal documentation and NYC-specific abbreviations: BIB ("brought in by"), NYPD, EMS, EDP ("emotionally disturbed person"), 70/10 (NYPD psychiatric pickup code), perp, ESU (NYPD Emergency Service Unit), FDNY (Fire Department of the City of New York; FDNY EMS operates NYC 911 ambulances, so "BIB FDNY" means EMS/ambulance transport, NOT law enforcement).

# Outcome categories

Assign exactly one of three transport-mode categories based on the documented transportation and custody at arrival.

ONLY POLICE
- Patient brought to the ED by law enforcement or in police custody, with no indication of EMS or ambulance transport.
- Qualifying language examples: "BIB NYPD", "under arrest", "in police custody" without ambulance involvement.

POLICE AND EMS
- Both law enforcement and EMS are involved in transport.
- Police must have ACTUALLY TRANSPORTED the patient OR been PHYSICALLY PRESENT at arrival.
- Police merely initiating the EMS call (e.g., "NYPD called EMS") does NOT qualify if police did not transport or accompany.
- Qualifying language examples: "BIB EMS with NYPD", police escort accompanying ambulance, both EMS/ambulance and police explicitly documented at arrival or triage.

NO POLICE
- No evidence of law enforcement involvement in transport or custody.
- Includes EMS-only transport and self / family / other transport.
- Edge case: if the patient is themselves a police officer (e.g., NYPD officer presenting for injury) but was not transported or detained by police, classify as NO POLICE.

Out of scope (assume not to occur; do NOT weight toward a police category):
- Post-arrival custody changes (patient arrives independently, later placed under arrest at bedside).
- Officer-as-patient transported by a fellow officer.

# Decision rules (apply in order; first match wins)

R1. Patient is identified as a police officer AND no mention of police transporting or accompanying this patient.
    -> transport_mode = no_police, rule_fired = officer_as_patient_no_transport

R2. Police called / notified EMS, but no documentation of police transporting the patient or being physically present at arrival.
    -> transport_mode = no_police, rule_fired = police_initiated_ems_only

R3. Both police AND EMS documented as transporting, OR police physically present at arrival / triage alongside EMS.
    -> transport_mode = police_and_ems
       rule_fired = co_transport_police_and_ems   (both transported)
       rule_fired = police_present_at_arrival_with_ems   (EMS transported, police present)

R4. Police transporting the patient WITHOUT any EMS involvement.
    -> transport_mode = only_police, rule_fired = transport_by_police_only

R5. EMS transport documented with no law enforcement mention.
    -> transport_mode = no_police, rule_fired = ems_only_transport

R6. Self / family / walk-in arrival, with no law enforcement involvement.
    -> transport_mode = no_police, rule_fired = self_or_other_transport

R7. No relevant arrival-mode or LEO language anywhere in the notes.
    -> transport_mode = no_police, rule_fired = no_leo_mention

R8. Documentation is genuinely contradictory across notes, or arrival mode is referenced but ambiguous.
    -> make best judgment for transport_mode, set confidence = low, rule_fired = ambiguous_documentation

# Confidence rubric

- high: arrival mode unambiguously documented with explicit transport-mode language.
- medium: arrival mode documented but with some ambiguity (e.g., "with police" — escort or just present?).
- low: arrival mode not documented, contradictory across notes, or based on indirect inference.

The confidence rating applies to transport_mode only.

# Custody context (secondary field)

Separately from transport mode, classify WHY police were involved at arrival. This is orthogonal to transport_mode: a handcuffed patient can arrive by ambulance (police_and_ems + criminal_custody).

- criminal_custody: arrest-related custody. Language: "under arrest", handcuffed, "prisoner", medical clearance for booking/arraignment, brought from precinct or corrections, officers guarding an arrestee.
- involuntary_psych_hold: involuntary psychiatric removal or hold. Language: EDP transport, "70/10", MHL 9.41 removal, "brought for involuntary psychiatric evaluation". IMPORTANT: officers may use custody-like language for these removals ("taken into custody for evaluation") — that is NOT criminal custody; classify it here.
- police_involved_no_custody: police transported or were present, but no custody or hold is documented (e.g., assault victim driven in by officers, officer standby without detention).
- not_applicable: REQUIRED when transport_mode = no_police.

If both criminal and psychiatric custody language appear, choose the one that better explains the arrival itself, and quote that language.

# Output

Return a JSON object that conforms exactly to the provided schema:
- transport_mode: one of "only_police", "police_and_ems", "no_police"
- rule_fired: one of the controlled-vocabulary values from R1–R8
- evidence_quote: short verbatim quote from the notes that drove the transport_mode decision (empty string only when rule_fired = no_leo_mention)
- confidence: "high", "medium", or "low" per the rubric above (transport_mode only)
- custody_context: one of "criminal_custody", "involuntary_psych_hold", "police_involved_no_custody", "not_applicable"
- custody_evidence_quote: short verbatim quote supporting custody_context (empty string when not_applicable or no directly relevant text)
"""


USER_PROMPT_TEMPLATE = """Encounter notes (chronological order):

{formatted_notes}

Classify this encounter.
"""


def format_notes(notes: Iterable[dict]) -> str:
    """Format a list of ED provider notes for inclusion in the user prompt.

    Each note dict must contain a 'text' field. Optional fields included in
    the section header if present: 'timestamp', 'role'.
    """
    notes = list(notes)
    n_total = len(notes)
    chunks = []
    for i, n in enumerate(notes, start=1):
        header_parts = [f"Note {i} of {n_total}"]
        if n.get("timestamp"):
            header_parts.append(str(n["timestamp"]))
        if n.get("role"):
            header_parts.append(str(n["role"]))
        header = " — ".join(header_parts)
        chunks.append(f"[{header}]\n{n['text'].strip()}")
    return "\n\n".join(chunks)


## 1. Imports + configuration

Config resolves in this order — use whichever is easiest on Minerva:

1. **Inline** (simplest): fill in `AZURE_OPENAI_ENDPOINT` / deployment / API version in the next cell. These are not secrets and may live in the notebook.
2. **`.env` file** in the working directory (template: `.env.example`), loaded automatically if present.
3. **The API key is special**: it is read from the environment/.env if set, otherwise the cell prompts for it interactively (`getpass`) so the key is **never stored in the notebook file** — the notebook is tracked in git, and a pasted key would leak to GitHub on the next commit.

Only four settings are used: key, endpoint, completion deployment, API version. `AZURE_OPENAI_MODEL` / `_EMBEDDING_DEPLOYMENT` from other templates are ignored (no embeddings in this pipeline).

In [ ]:
import hashlib
import json
import os
import random
import time
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd
from openai import (
    APIConnectionError,
    APITimeoutError,
    AzureOpenAI,
    RateLimitError,
)

from classifier_lib import (
    ChartClassification,
    PROMPT_VERSION,
    PROMPT_VERSION_NOTES,
    SYSTEM_PROMPT,
    USER_PROMPT_TEMPLATE,
    format_notes,
)

# ── Inline config: fill these in (NOT secrets — safe to keep in the notebook).
# Leave blank to fall back to .env / environment variables instead.
INLINE_AZURE_OPENAI_ENDPOINT = ""  # e.g. "https://<resource>.openai.azure.com/"
INLINE_AZURE_OPENAI_COMPLETION_DEPLOYMENT = "gpt-5-2025-08-07"
INLINE_AZURE_OPENAI_API_VERSION = "2024-08-01-preview"

# Optional .env fallback (works if python-dotenv is installed and a .env
# file exists in the working directory; harmless otherwise).
try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass


def _clean(v: str) -> str:
    # strip whitespace and stray straight/smart quotes from pasted values
    return (v or "").strip().strip("'\"“”")


def _resolve(inline_value: str, env_name: str) -> str:
    return _clean(inline_value) or _clean(os.environ.get(env_name, ""))


config = {
    "AZURE_OPENAI_ENDPOINT": _resolve(
        INLINE_AZURE_OPENAI_ENDPOINT, "AZURE_OPENAI_ENDPOINT"
    ),
    "AZURE_OPENAI_COMPLETION_DEPLOYMENT": _resolve(
        INLINE_AZURE_OPENAI_COMPLETION_DEPLOYMENT,
        "AZURE_OPENAI_COMPLETION_DEPLOYMENT",
    ),
    "AZURE_OPENAI_API_VERSION": _resolve(
        INLINE_AZURE_OPENAI_API_VERSION, "AZURE_OPENAI_API_VERSION"
    ),
}
missing = [k for k, v in config.items() if not v]
if missing:
    raise RuntimeError(
        f"Missing config: {missing}. Fill in the INLINE_ values above or "
        "provide them via .env (see .env.example)."
    )

# The endpoint must be a full URL. A bare hostname (missing scheme) makes
# httpx raise UnsupportedProtocol, so prepend https:// if needed.
_ep = config["AZURE_OPENAI_ENDPOINT"]
if not _ep.lower().startswith(("http://", "https://")):
    _ep = "https://" + _ep
    print(f"note: endpoint had no scheme; using {_ep}")
if ".openai.azure.com" not in _ep.lower() and ".cognitiveservices.azure.com" not in _ep.lower():
    print(
        "WARNING: endpoint does not look like an Azure OpenAI URL "
        "(expected https://<resource>.openai.azure.com/) — double-check it."
    )
config["AZURE_OPENAI_ENDPOINT"] = _ep


# API key: environment/.env if set, otherwise interactive prompt.
# NEVER paste the key into this cell — the notebook is tracked in git.
def _looks_like_key(v: str) -> bool:
    # Azure keys are long alphanumeric strings — not URLs, not short.
    return len(v) >= 20 and "/" not in v and "." not in v and " " not in v


api_key = _clean(os.environ.get("AZURE_OPENAI_API_KEY", ""))
if api_key and not _looks_like_key(api_key):
    print(
        "Ignoring AZURE_OPENAI_API_KEY from environment — it does not look "
        "like an API key (URLs/dots/slashes are not keys)."
    )
    api_key = ""
for _attempt in range(3):
    if api_key:
        break
    api_key = _clean(getpass("AZURE_OPENAI_API_KEY (input hidden, not saved): "))
    if not _looks_like_key(api_key):
        print(
            "That does not look like an Azure API key (expect a long "
            "alphanumeric string from Keys and Endpoint -> Key 1). Try again."
        )
        api_key = ""
if not api_key:
    raise RuntimeError("No valid API key provided.")
config["AZURE_OPENAI_API_KEY"] = api_key

# Export so downstream cells (client init, metadata) read one source.
os.environ.update(config)

print(f"PROMPT_VERSION = {PROMPT_VERSION}")
print(f"endpoint       = {config['AZURE_OPENAI_ENDPOINT']}")
print(f"deployment     = {config['AZURE_OPENAI_COMPLETION_DEPLOYMENT']}")
print(f"api_version    = {config['AZURE_OPENAI_API_VERSION']}")
print(f"api_key        = ****{api_key[-4:]} (hidden)")

## 2. Run constants

Edit these for each run. The split seed should be **set once and never changed** after the first dev run; changing it reshuffles the dev/eval assignment and invalidates pre-registration.

In [ ]:
# Data directory: honors $DATA_DIR; otherwise uses ./data if it exists,
# else the notebook's own folder (files sitting flat next to the .ipynb).
_env_data = os.environ.get("DATA_DIR", "")
DATA_DIR = Path(_env_data) if _env_data else (Path("./data") if Path("./data").exists() else Path("."))
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "./outputs"))
# Notes file: full LEO cohort (~3,097 encounters), one row per note CHUNK.
NOTES_FILE = os.environ.get("NOTES_FILE", "full_cohort_notes.csv")
# Boilerplate note types excluded before classification (no arrival-mode
# signal, pure cost): patient handouts and billing attestations.
NOTE_TYPE_EXCLUDE = {"Discharge Instructions", "Attestation"}
# The primary adjudicator's abstraction file (the 10% sample): provides
# EncounterKey + the adjudicated "LEO-Transport?" label. Its Keywords
# column is NEVER exposed to the LLM (see docs/approach.md).
# NOTE: use the _updated_ file, not the older police_random_10pct.csv.
GOLD_STANDARD_FILE = os.environ.get(
    "GOLD_STANDARD_FILE", "police_random_10pct_updated_07_08_26.csv"
)
# CSV/XLSX listing the double-coded (secondary-adjudicator overlap)
# EncounterKeys — the kappa-overlap list (62 charts). Excluded from the
# dev-eligible pool so the full overlap lands in the eval set.
DOUBLE_CODED_FILE = os.environ.get("DOUBLE_CODED_FILE", "kappa_overlap_list.csv")
# Column in the gold-standard file holding the adjudicated 0/1/2 code.
# Used ONLY to stratify the dev draw so every category has enough charts
# for prompt iteration; the label never reaches the LLM. The split cell
# raises with the real column list if this name is wrong.
GOLD_LABEL_COL = os.environ.get("GOLD_LABEL_COL", "LEO-Transport?")
MIN_PER_CATEGORY = 10  # dev-set floor per gold-label category

# Split parameters. CHANGE SEED ONLY ONCE.
SPLIT_SEED = 20260524  # the date the split was first generated; lock forever
DEV_N = 50
# the primary adjudicator's master file has 312 rows; probe (2026-07-08) found 1 null
# label and 2 encounters without notes, so the realized universe is
# ~309. The runner prints the exact accounting.
EXPECTED_VALIDATION_N = 312

# EVAL LOCK. Leave False for dev scoring (dev 50 only). Set True ONLY for
# the final, one-shot held-out evaluation with the LOCKED prompt — it
# scores the full validation universe (dev + eval = ~309). The prompt is
# LOCKED at v1.0 (v1.1 was tried on the dev set and reverted); do not
# change the prompt after setting this True.
UNLOCK_EVAL = False

# Cost ceiling for THIS invocation (USD). Dollar figures are NOMINAL
# (placeholder pricing, flagged unverified); the guard is a runaway
# backstop, not a real budget. Raise it (e.g., 500) before the ~309-chart
# eval run so it does not abort a legitimate run.
COST_CEILING_USD = float(os.environ.get("COST_CEILING_USD", "5.00"))

# Pricing for the deployed model (USD per 1M tokens). NOMINAL / unverified.
INPUT_USD_PER_M = 10.00
OUTPUT_USD_PER_M = 30.00

# Token budget per call (reasoning models consume reasoning + visible tokens).
MAX_COMPLETION_TOKENS = 4096

# Retry policy
MAX_RETRIES = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR   = {DATA_DIR.resolve()}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")
print(f"notes file = {NOTES_FILE} (excluding types: {sorted(NOTE_TYPE_EXCLUDE)})")
print(f"gold file  = {GOLD_STANDARD_FILE}")
print(f"dev n = {DEV_N}, seed = {SPLIT_SEED}, stratify by = {GOLD_LABEL_COL or '(unset — plain random draw)'}")
print(f"UNLOCK_EVAL = {UNLOCK_EVAL}  (True = score full held-out set with LOCKED prompt)")
print(f"cost cap   = ${COST_CEILING_USD:.2f} (nominal)")

## 3. Azure client + strict schema

The OpenAI structured-outputs strict mode requires `additionalProperties: false` and all properties marked `required` on every object in the schema. The helper below enforces this recursively against Pydantic's `model_json_schema()` output.

In [ ]:
client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)
DEPLOYMENT = os.environ["AZURE_OPENAI_COMPLETION_DEPLOYMENT"]

import copy


def _inline_refs(schema, defs):
    """Inline $ref pointers. OpenAI strict mode rejects $ref with sibling
    keywords, and Pydantic emits enum fields as $ref + description."""
    if isinstance(schema, dict):
        if "$ref" in schema:
            target = copy.deepcopy(defs[schema["$ref"].split("/")[-1]])
            extras = {k: v for k, v in schema.items() if k != "$ref"}
            target.update(extras)  # field-level description wins
            return _inline_refs(target, defs)
        return {k: _inline_refs(v, defs) for k, v in schema.items()}
    if isinstance(schema, list):
        return [_inline_refs(x, defs) for x in schema]
    return schema


def _enforce_strict(schema):
    """Recursively enforce OpenAI strict-mode constraints on a JSON Schema."""
    if isinstance(schema, dict):
        if schema.get("type") == "object" and "properties" in schema:
            schema["additionalProperties"] = False
            schema["required"] = list(schema["properties"].keys())
        for v in schema.values():
            if isinstance(v, (dict, list)):
                _enforce_strict(v)
    elif isinstance(schema, list):
        for item in schema:
            if isinstance(item, (dict, list)):
                _enforce_strict(item)
    return schema


_raw_schema = ChartClassification.model_json_schema()
_defs = _raw_schema.pop("$defs", {})
STRICT_SCHEMA = _enforce_strict(_inline_refs(_raw_schema, _defs))
assert "$ref" not in json.dumps(STRICT_SCHEMA), "unresolved $ref in schema"
print(json.dumps(STRICT_SCHEMA, indent=2)[:1200], "...")

In [ ]:
# Connectivity smoke test: one minimal call to verify the key, endpoint,
# deployment name, and Minerva's outbound network before touching data.
# Costs a fraction of a cent. If this fails with a connection timeout,
# it's a Minerva network/proxy issue, not a code problem.
_ping = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[{"role": "user", "content": "Reply with the single word: ok"}],
    max_completion_tokens=200,
)
print("smoke test reply:", _ping.choices[0].message.content)
print(
    f"tokens: prompt={_ping.usage.prompt_tokens}, "
    f"completion={_ping.usage.completion_tokens}"
)
print("Azure OpenAI connectivity OK — safe to proceed.")

def load_notes_table(path: Path) -> pd.DataFrame:
    """Load csv/xlsx/parquet; CSVs try utf-8 then Windows encodings
    (Excel exports often contain cp1252 bytes like 0xa0)."""
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if path.suffix.lower() == ".csv":
        for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
            try:
                df = pd.read_csv(path, encoding=enc)
                if enc != "utf-8":
                    print(f"({path.name}: decoded with {enc})")
                return df
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError("all", b"", 0, 1, f"could not decode {path}")
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported notes file format: {path.suffix}")


notes_path = DATA_DIR / NOTES_FILE
if not notes_path.exists():
    raise FileNotFoundError(
        f"Notes file not found at {notes_path}. Place the note-pull file in "
        "DATA_DIR or set NOTES_FILE."
    )

notes_df = load_notes_table(notes_path)
required_cols = {"EncounterKey", "NOTE_TEXT"}
missing_cols = required_cols - set(notes_df.columns)
if missing_cols:
    raise ValueError(f"Notes file missing required columns: {missing_cols}")

# Drop boilerplate note types (patient handouts, billing attestations).
if NOTE_TYPE_EXCLUDE and "NOTE_TYPE" in notes_df.columns:
    _excl = notes_df["NOTE_TYPE"].isin(NOTE_TYPE_EXCLUDE)
    print(f"excluding {int(_excl.sum())} of {len(notes_df)} note rows "
          f"by NOTE_TYPE {sorted(NOTE_TYPE_EXCLUDE)}")
    notes_df = notes_df[~_excl]

if "NOTE_ID" in notes_df.columns:
    # Long notes are CHUNKED across consecutive rows sharing a NOTE_ID
    # (probe: 4,745 of 23,538 NOTE_IDs span >1 row; chunks never cross
    # encounters). Stitch chunks into one note per NOTE_ID, in order.
    n_rows_before = len(notes_df)
    notes_df = notes_df.reset_index().rename(columns={"index": "_row_order"})
    notes_df["NOTE_TEXT"] = notes_df["NOTE_TEXT"].fillna("").astype(str)
    agg_spec = {
        "NOTE_TEXT": ("NOTE_TEXT", "\n".join),
        "_row_order": ("_row_order", "min"),
    }
    if "NOTE_TYPE" in notes_df.columns:
        agg_spec["NOTE_TYPE"] = ("NOTE_TYPE", "first")
    notes_df = (
        notes_df.sort_values(["EncounterKey", "NOTE_ID", "_row_order"])
        .groupby(["EncounterKey", "NOTE_ID"], as_index=False, sort=False)
        .agg(**agg_spec)
        .sort_values(["EncounterKey", "NOTE_ID"])
        .reset_index(drop=True)
    )
    print(
        f"stitched {n_rows_before} rows into {len(notes_df)} notes "
        f"(chunked rows joined on NOTE_ID within encounter)"
    )
else:
    print(
        "WARNING: no NOTE_ID column — keeping the file's row order as the "
        "within-encounter chronological order."
    )
    notes_df = notes_df.reset_index(drop=True)
print(f"{len(notes_df)} notes across {notes_df['EncounterKey'].nunique()} encounters")


def encounter_to_notes(encounter_df: pd.DataFrame) -> list[dict]:
    return [
        {"text": row["NOTE_TEXT"], "role": row.get("NOTE_TYPE")}
        for _, row in encounter_df.iterrows()
        if isinstance(row["NOTE_TEXT"], str) and row["NOTE_TEXT"].strip()
    ]


encounters_by_key = {
    key: encounter_to_notes(group)
    for key, group in notes_df.groupby("EncounterKey", sort=False)
}

# The notes file holds the full LEO cohort (~3,100 encounters). The
# dev/eval split applies only to the adjudicated validation universe,
# so restrict to EncounterKeys present in the gold-standard master file.
gold_path = DATA_DIR / GOLD_STANDARD_FILE
if not gold_path.exists():
    raise FileNotFoundError(
        f"Gold-standard file not found at {gold_path}. The dev runner refuses "
        "to split without it — otherwise the 50/262 split would be drawn from "
        "the wrong universe. Set GOLD_STANDARD_FILE."
    )
gold_df = load_notes_table(gold_path)
if "EncounterKey" not in gold_df.columns:
    raise ValueError(
        f"Gold-standard file has no 'EncounterKey' column (found: {list(gold_df.columns)}). "
        "Run probe_data.ipynb to confirm the real key column, then adjust."
    )
n_gold_rows = len(gold_df)
# Unlabeled charts cannot be validated or used for stratification.
if GOLD_LABEL_COL and GOLD_LABEL_COL in gold_df.columns:
    n_null_label = int(gold_df[GOLD_LABEL_COL].isna().sum())
    if n_null_label:
        print(f"dropping {n_null_label} gold row(s) with null '{GOLD_LABEL_COL}' (cannot validate unlabeled charts)")
        gold_df = gold_df.dropna(subset=[GOLD_LABEL_COL])
gold_keys = set(gold_df["EncounterKey"].dropna().unique())
print(f"gold-standard file: {n_gold_rows} rows -> {len(gold_keys)} labeled unique EncounterKeys")

all_keys = sorted(set(encounters_by_key) & gold_keys)
n_gold_without_notes = len(gold_keys - set(encounters_by_key))
print(f"validation universe (labeled gold ∩ notes): {len(all_keys)} encounters")
if n_gold_without_notes:
    print(f"NOTE: {n_gold_without_notes} labeled gold encounter(s) have no notes in the notes file (excluded)")
if abs(len(all_keys) - EXPECTED_VALIDATION_N) > 5:
    print(
        f"WARNING: validation universe is {len(all_keys)}, far from expected "
        f"~{EXPECTED_VALIDATION_N}. Investigate before trusting the split."
    )

In [ ]:
def load_notes_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported notes file format: {path.suffix}")


notes_path = DATA_DIR / NOTES_FILE
if not notes_path.exists():
    raise FileNotFoundError(
        f"Notes file not found at {notes_path}. Place the populated full-cohort "
        "shell in DATA_DIR or set NOTES_FILE."
    )

notes_df = load_notes_table(notes_path)
required_cols = {"EncounterKey", "NOTE_TEXT", "NOTE_ID"}
missing_cols = required_cols - set(notes_df.columns)
if missing_cols:
    raise ValueError(f"Notes file missing required columns: {missing_cols}")

notes_df = notes_df.sort_values(["EncounterKey", "NOTE_ID"]).reset_index(drop=True)
print(f"loaded {len(notes_df)} note rows across {notes_df['EncounterKey'].nunique()} encounters")


def encounter_to_notes(encounter_df: pd.DataFrame) -> list[dict]:
    return [
        {"text": row["NOTE_TEXT"], "role": row.get("NOTE_TYPE")}
        for _, row in encounter_df.iterrows()
        if isinstance(row["NOTE_TEXT"], str) and row["NOTE_TEXT"].strip()
    ]


encounters_by_key = {
    key: encounter_to_notes(group)
    for key, group in notes_df.groupby("EncounterKey")
}

# The notes file is the FULL-COHORT shell (may hold all ~4,000 encounters).
# The dev/eval split applies only to the adjudicated validation universe,
# so restrict to EncounterKeys present in the gold-standard master file.
gold_path = DATA_DIR / GOLD_STANDARD_FILE
if not gold_path.exists():
    raise FileNotFoundError(
        f"Gold-standard file not found at {gold_path}. The dev runner refuses "
        "to split without it — otherwise the 50/262 split would be drawn from "
        "the wrong universe. Set GOLD_STANDARD_FILE in .env."
    )
gold_df = load_notes_table(gold_path)
if "EncounterKey" not in gold_df.columns:
    raise ValueError(
        f"Gold-standard file has no 'EncounterKey' column (found: {list(gold_df.columns)}). "
        "Run probe_data.ipynb to confirm the real key column, then adjust."
    )
gold_keys = set(gold_df["EncounterKey"].dropna().unique())
print(f"gold-standard file: {len(gold_df)} rows, {len(gold_keys)} unique EncounterKeys")

all_keys = sorted(set(encounters_by_key) & gold_keys)
n_gold_without_notes = len(gold_keys - set(encounters_by_key))
print(f"validation universe (notes ∩ gold): {len(all_keys)} encounters")
if n_gold_without_notes:
    print(f"WARNING: {n_gold_without_notes} gold-standard encounters have no notes in the notes file")
if len(all_keys) != EXPECTED_VALIDATION_N:
    print(
        f"WARNING: validation universe is {len(all_keys)}, expected "
        f"{EXPECTED_VALIDATION_N}. Investigate before trusting the split."
    )

double_coded_keys = set()
if DOUBLE_CODED_FILE:
    dc_path = DATA_DIR / DOUBLE_CODED_FILE
    if not dc_path.exists():
        raise FileNotFoundError(
            f"DOUBLE_CODED_FILE is set but {dc_path} does not exist."
        )
    dc_df = load_notes_table(dc_path)
    if "EncounterKey" not in dc_df.columns:
        raise ValueError(
            f"Double-coded file has no 'EncounterKey' column (found: {list(dc_df.columns)})."
        )
    dc_keys_raw = set(dc_df["EncounterKey"].dropna().unique())
    double_coded_keys = dc_keys_raw & set(all_keys)
    print(
        f"double-coded overlap: {len(double_coded_keys)} encounters "
        f"(of {len(dc_keys_raw)} listed) forced into eval"
    )
    if len(dc_keys_raw - double_coded_keys):
        print(
            f"WARNING: {len(dc_keys_raw - double_coded_keys)} double-coded keys "
            "not found in the validation universe"
        )
else:
    print(
        "WARNING: DOUBLE_CODED_FILE not set. The design requires the 62 "
        "double-coded charts to stay in eval; without the key list the dev "
        "draw may include them. Get the list before the first real dev run."
    )

# Gold labels for stratification (never shown to the LLM).
gold_labels = {}
if GOLD_LABEL_COL:
    if GOLD_LABEL_COL not in gold_df.columns:
        raise ValueError(
            f"GOLD_LABEL_COL='{GOLD_LABEL_COL}' not in gold-standard columns "
            f"({list(gold_df.columns)}). Check probe_data.ipynb output."
        )
    _label_series = (
        gold_df.dropna(subset=["EncounterKey"])
        .set_index("EncounterKey")[GOLD_LABEL_COL]
    )
    try:
        _label_series = _label_series.astype(int)  # 1.0 -> 1 for clean reporting
    except (TypeError, ValueError):
        pass
    gold_labels = _label_series.to_dict()
else:
    print(
        "WARNING: GOLD_LABEL_COL not set — dev draw will be plain random, "
        "so the rare category may be underrepresented in the dev set."
    )


def stratified_dev_draw(pool: list, labels: dict, dev_n: int, seed: int, floor: int) -> set:
    """Seeded dev draw: `floor` charts per label category first, then fill
    the remaining slots at random from the rest of the pool. With no
    labels, degrades to a plain seeded random draw."""
    rng = random.Random(seed)
    dev = set()
    leftover = []
    by_label = {}
    for k in sorted(pool):
        by_label.setdefault(labels.get(k, "__unlabeled__"), []).append(k)
    for label in sorted(by_label, key=str):
        keys = by_label[label]
        rng.shuffle(keys)
        take = min(floor, len(keys), dev_n - len(dev)) if labels else 0
        dev.update(keys[:take])
        leftover.extend(keys[take:])
    leftover.sort()
    rng.shuffle(leftover)
    dev.update(leftover[: dev_n - len(dev)])
    return dev


dev_pool = [k for k in all_keys if k not in double_coded_keys]
dev_keys = stratified_dev_draw(
    dev_pool, gold_labels, DEV_N, SPLIT_SEED, MIN_PER_CATEGORY
)
eval_keys = set(all_keys) - dev_keys
print(f"dev:  {len(dev_keys)} encounters (drawn from pool of {len(dev_pool)})")
print(f"eval: {len(eval_keys)} encounters incl. all double-coded (LOCKED unless UNLOCK_EVAL=True)")
if gold_labels:
    dev_label_counts = pd.Series(
        [gold_labels.get(k) for k in dev_keys]
    ).value_counts(dropna=False).to_dict()
    print(f"dev-set gold-label counts: {dev_label_counts}")
else:
    dev_label_counts = None

split_records = (
    [{"EncounterKey": k, "split": "dev"} for k in sorted(dev_keys)]
    + [{"EncounterKey": k, "split": "eval"} for k in sorted(eval_keys)]
)
split_path = OUTPUT_DIR / f"dev_eval_split_seed{SPLIT_SEED}.csv"
pd.DataFrame(split_records).to_csv(split_path, index=False)
print(f"wrote {split_path}")

if UNLOCK_EVAL:
    print("WARNING: UNLOCK_EVAL=True. This run will score eval encounters.")
    target_keys = sorted(dev_keys | eval_keys)
else:
    target_keys = sorted(dev_keys)
print(f"will score {len(target_keys)} encounters this run")

In [ ]:
double_coded_keys = set()
if DOUBLE_CODED_FILE:
    dc_path = DATA_DIR / DOUBLE_CODED_FILE
    if not dc_path.exists():
        raise FileNotFoundError(
            f"DOUBLE_CODED_FILE is set but {dc_path} does not exist."
        )
    dc_df = load_notes_table(dc_path)
    if "EncounterKey" not in dc_df.columns:
        raise ValueError(
            f"Double-coded file has no 'EncounterKey' column (found: {list(dc_df.columns)})."
        )
    dc_keys_raw = set(dc_df["EncounterKey"].dropna().unique())
    double_coded_keys = dc_keys_raw & set(all_keys)
    print(
        f"double-coded overlap: {len(double_coded_keys)} encounters "
        f"(of {len(dc_keys_raw)} listed) forced into eval"
    )
    if len(dc_keys_raw - double_coded_keys):
        print(
            f"WARNING: {len(dc_keys_raw - double_coded_keys)} double-coded keys "
            "not found in the validation universe"
        )
else:
    print(
        "WARNING: DOUBLE_CODED_FILE not set. The design requires the 62 "
        "double-coded charts to stay in eval; without the key list the dev "
        "draw may include them. Get the list before the first real dev run."
    )

# Gold labels for stratification (never shown to the LLM).
gold_labels = {}
if GOLD_LABEL_COL:
    if GOLD_LABEL_COL not in gold_df.columns:
        raise ValueError(
            f"GOLD_LABEL_COL='{GOLD_LABEL_COL}' not in gold-standard columns "
            f"({list(gold_df.columns)}). Check probe_data.ipynb output."
        )
    gold_labels = (
        gold_df.dropna(subset=["EncounterKey"])
        .set_index("EncounterKey")[GOLD_LABEL_COL]
        .to_dict()
    )
else:
    print(
        "WARNING: GOLD_LABEL_COL not set — dev draw will be plain random, "
        "so the rare category may be underrepresented in the dev set."
    )


def stratified_dev_draw(pool: list, labels: dict, dev_n: int, seed: int, floor: int) -> set:
    """Seeded dev draw: `floor` charts per label category first, then fill
    the remaining slots at random from the rest of the pool. With no
    labels, degrades to a plain seeded random draw."""
    rng = random.Random(seed)
    dev = set()
    leftover = []
    by_label = {}
    for k in sorted(pool):
        by_label.setdefault(labels.get(k, "__unlabeled__"), []).append(k)
    for label in sorted(by_label, key=str):
        keys = by_label[label]
        rng.shuffle(keys)
        take = min(floor, len(keys), dev_n - len(dev)) if labels else 0
        dev.update(keys[:take])
        leftover.extend(keys[take:])
    leftover.sort()
    rng.shuffle(leftover)
    dev.update(leftover[: dev_n - len(dev)])
    return dev


dev_pool = [k for k in all_keys if k not in double_coded_keys]
dev_keys = stratified_dev_draw(
    dev_pool, gold_labels, DEV_N, SPLIT_SEED, MIN_PER_CATEGORY
)
eval_keys = set(all_keys) - dev_keys
print(f"dev:  {len(dev_keys)} encounters (drawn from pool of {len(dev_pool)})")
print(f"eval: {len(eval_keys)} encounters incl. all double-coded (LOCKED unless UNLOCK_EVAL=True)")
if gold_labels:
    dev_label_counts = pd.Series(
        [gold_labels.get(k) for k in dev_keys]
    ).value_counts(dropna=False).to_dict()
    print(f"dev-set gold-label counts: {dev_label_counts}")
else:
    dev_label_counts = None

split_records = (
    [{"EncounterKey": k, "split": "dev"} for k in sorted(dev_keys)]
    + [{"EncounterKey": k, "split": "eval"} for k in sorted(eval_keys)]
)
split_path = OUTPUT_DIR / f"dev_eval_split_seed{SPLIT_SEED}.csv"
pd.DataFrame(split_records).to_csv(split_path, index=False)
print(f"wrote {split_path}")

if UNLOCK_EVAL:
    print("WARNING: UNLOCK_EVAL=True. This run will score eval encounters.")
    target_keys = sorted(dev_keys | eval_keys)
else:
    target_keys = sorted(dev_keys)
print(f"will score {len(target_keys)} encounters this run")

## 6. Per-encounter classify call with retry

Single call per encounter (per v1.0 design decision). Retries on `RateLimitError`, `APIConnectionError`, `APITimeoutError` with exponential backoff + jitter. GPT-5 reasoning-model conventions: no temperature; `max_completion_tokens` covers reasoning + visible output.

In [ ]:
def classify_encounter(notes: list[dict]) -> tuple[ChartClassification, int, int]:
    """Classify one encounter. Returns (parsed, prompt_tokens, completion_tokens)."""
    user_msg = USER_PROMPT_TEMPLATE.format(formatted_notes=format_notes(notes))
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=DEPLOYMENT,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "ChartClassification",
                        "strict": True,
                        "schema": STRICT_SCHEMA,
                    },
                },
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )
            raw = response.choices[0].message.content
            parsed = ChartClassification.model_validate_json(raw)
            return parsed, response.usage.prompt_tokens, response.usage.completion_tokens
        except (RateLimitError, APIConnectionError, APITimeoutError) as e:
            last_exc = e
            wait = (2 ** attempt) + random.random()
            print(f"  retry {attempt + 1}/{MAX_RETRIES} after {wait:.1f}s: {type(e).__name__}")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {MAX_RETRIES} retries: {last_exc}")


def estimate_tokens(text: str) -> int:
    """Rough character-based estimate for pre-flight cost projection."""
    return max(1, len(text) // 4)

## 7. Main loop

Classifies each target encounter, accumulates token + cost stats, and aborts cleanly if projected next-call cost would push total above `COST_CEILING_USD`.

In [ ]:
predictions = []
errors = []
total_prompt_tokens = 0
total_completion_tokens = 0
cumulative_cost = 0.0
aborted_for_cost = False

run_started_at = datetime.now(timezone.utc).isoformat()

for i, encounter_key in enumerate(target_keys, start=1):
    notes = encounters_by_key[encounter_key]
    if not notes:
        errors.append({"encounter_key": encounter_key, "error": "no usable notes"})
        print(f"[{i}/{len(target_keys)}] {encounter_key}: SKIP (no notes)")
        continue

    user_msg = USER_PROMPT_TEMPLATE.format(formatted_notes=format_notes(notes))
    est_prompt_tokens = estimate_tokens(SYSTEM_PROMPT) + estimate_tokens(user_msg)
    est_next_cost = (
        est_prompt_tokens * INPUT_USD_PER_M / 1e6
        + MAX_COMPLETION_TOKENS * OUTPUT_USD_PER_M / 1e6
    )
    if cumulative_cost + est_next_cost > COST_CEILING_USD:
        aborted_for_cost = True
        print(
            f"ABORT: projected total ${cumulative_cost + est_next_cost:.2f} "
            f"exceeds ceiling ${COST_CEILING_USD:.2f} at encounter {i}/{len(target_keys)}"
        )
        break

    try:
        parsed, p_tok, c_tok = classify_encounter(notes)
        cost = p_tok * INPUT_USD_PER_M / 1e6 + c_tok * OUTPUT_USD_PER_M / 1e6
        total_prompt_tokens += p_tok
        total_completion_tokens += c_tok
        cumulative_cost += cost
        predictions.append(
            {
                "encounter_key": encounter_key,
                "transport_mode": parsed.transport_mode.value,
                "rule_fired": parsed.rule_fired.value,
                "evidence_quote": parsed.evidence_quote,
                "confidence": parsed.confidence,
                "custody_context": parsed.custody_context.value,
                "custody_evidence_quote": parsed.custody_evidence_quote,
                "prompt_tokens": p_tok,
                "completion_tokens": c_tok,
                "cost_usd": round(cost, 6),
            }
        )
        print(
            f"[{i}/{len(target_keys)}] {encounter_key}: "
            f"{parsed.transport_mode.value} | {parsed.rule_fired.value} | "
            f"custody={parsed.custody_context.value} | "
            f"conf={parsed.confidence} | total=${cumulative_cost:.3f}"
        )
    except Exception as e:
        errors.append({"encounter_key": encounter_key, "error": repr(e)})
        print(f"[{i}/{len(target_keys)}] {encounter_key}: ERROR {type(e).__name__}: {e}")

run_ended_at = datetime.now(timezone.utc).isoformat()
print(
    f"\nfinished. predictions={len(predictions)} errors={len(errors)} "
    f"aborted_for_cost={aborted_for_cost} total_cost=${cumulative_cost:.3f}"
)

import numpy as np


def _san(o):
    """Recursively convert numpy scalars/arrays to native Python types so
    json can serialize them (EncounterKey etc. arrive as numpy int64)."""
    if isinstance(o, dict):
        return {_san(k): _san(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_san(x) for x in o]
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o


stamp = run_started_at.replace(":", "").replace("-", "")
pred_path = OUTPUT_DIR / f"predictions_{PROMPT_VERSION}_{stamp}.jsonl"
meta_path = OUTPUT_DIR / f"metadata_{PROMPT_VERSION}_{stamp}.json"
err_path = OUTPUT_DIR / f"errors_{PROMPT_VERSION}_{stamp}.json"

with pred_path.open("w") as f:
    for row in predictions:
        f.write(json.dumps(_san(row)) + "\n")

metadata = _san({
    "prompt_version": PROMPT_VERSION,
    "deployment": DEPLOYMENT,
    "api_version": os.environ["AZURE_OPENAI_API_VERSION"],
    "run_started_at": run_started_at,
    "run_ended_at": run_ended_at,
    "split_seed": SPLIT_SEED,
    "dev_n": DEV_N,
    "n_validation_universe": len(all_keys),
    "expected_validation_n": EXPECTED_VALIDATION_N,
    "n_double_coded_forced_eval": len(double_coded_keys),
    "double_coded_file": DOUBLE_CODED_FILE or None,
    "gold_label_col": GOLD_LABEL_COL or None,
    "min_per_category": MIN_PER_CATEGORY if GOLD_LABEL_COL else None,
    "dev_label_counts": dev_label_counts if dev_label_counts else None,
    "unlock_eval": UNLOCK_EVAL,
    "n_target": len(target_keys),
    "n_predictions": len(predictions),
    "n_errors": len(errors),
    "aborted_for_cost": aborted_for_cost,
    "total_prompt_tokens": total_prompt_tokens,
    "total_completion_tokens": total_completion_tokens,
    "total_cost_usd": round(cumulative_cost, 6),
    "cost_ceiling_usd": COST_CEILING_USD,
    "input_usd_per_million": INPUT_USD_PER_M,
    "output_usd_per_million": OUTPUT_USD_PER_M,
    "max_completion_tokens": MAX_COMPLETION_TOKENS,
    "max_retries": MAX_RETRIES,
    "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest(),
    "schema_sha256": hashlib.sha256(
        json.dumps(STRICT_SCHEMA, sort_keys=True).encode()
    ).hexdigest(),
    "prompt_version_notes": PROMPT_VERSION_NOTES,
})
with meta_path.open("w") as f:
    json.dump(metadata, f, indent=2)

if errors:
    with err_path.open("w") as f:
        json.dump(_san(errors), f, indent=2)
    print(f"errors written to {err_path}")

print(f"predictions -> {pred_path}")
print(f"metadata    -> {meta_path}")

In [ ]:
stamp = run_started_at.replace(":", "").replace("-", "")
pred_path = OUTPUT_DIR / f"predictions_{PROMPT_VERSION}_{stamp}.jsonl"
meta_path = OUTPUT_DIR / f"metadata_{PROMPT_VERSION}_{stamp}.json"
err_path = OUTPUT_DIR / f"errors_{PROMPT_VERSION}_{stamp}.json"

with pred_path.open("w") as f:
    for row in predictions:
        f.write(json.dumps(row) + "\n")

metadata = {
    "prompt_version": PROMPT_VERSION,
    "deployment": DEPLOYMENT,
    "api_version": os.environ["AZURE_OPENAI_API_VERSION"],
    "run_started_at": run_started_at,
    "run_ended_at": run_ended_at,
    "split_seed": SPLIT_SEED,
    "dev_n": DEV_N,
    "n_validation_universe": len(all_keys),
    "expected_validation_n": EXPECTED_VALIDATION_N,
    "n_double_coded_forced_eval": len(double_coded_keys),
    "double_coded_file": DOUBLE_CODED_FILE or None,
    "gold_label_col": GOLD_LABEL_COL or None,
    "min_per_category": MIN_PER_CATEGORY if GOLD_LABEL_COL else None,
    "dev_label_counts": (
        {str(k): int(v) for k, v in dev_label_counts.items()}
        if dev_label_counts
        else None
    ),
    "unlock_eval": UNLOCK_EVAL,
    "n_target": len(target_keys),
    "n_predictions": len(predictions),
    "n_errors": len(errors),
    "aborted_for_cost": aborted_for_cost,
    "total_prompt_tokens": total_prompt_tokens,
    "total_completion_tokens": total_completion_tokens,
    "total_cost_usd": round(cumulative_cost, 6),
    "cost_ceiling_usd": COST_CEILING_USD,
    "input_usd_per_million": INPUT_USD_PER_M,
    "output_usd_per_million": OUTPUT_USD_PER_M,
    "max_completion_tokens": MAX_COMPLETION_TOKENS,
    "max_retries": MAX_RETRIES,
    "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest(),
    "schema_sha256": hashlib.sha256(
        json.dumps(STRICT_SCHEMA, sort_keys=True).encode()
    ).hexdigest(),
    "prompt_version_notes": PROMPT_VERSION_NOTES,
}
with meta_path.open("w") as f:
    json.dump(metadata, f, indent=2)

if errors:
    with err_path.open("w") as f:
        json.dump(errors, f, indent=2)
    print(f"errors written to {err_path}")

print(f"predictions -> {pred_path}")
print(f"metadata    -> {meta_path}")

## 9. Quick summary

Per-category counts and per-rule counts on what was just classified. For PPV computation, merge with the gold-standard labels in a separate evaluation notebook (forthcoming).

In [ ]:
if predictions:
    pred_df = pd.DataFrame(predictions)
    print("transport_mode counts:")
    print(pred_df["transport_mode"].value_counts().to_string())
    print("\nrule_fired counts:")
    print(pred_df["rule_fired"].value_counts().to_string())
    print("\ncustody_context counts (EXPLORATORY — not human-validated):")
    print(pred_df["custody_context"].value_counts().to_string())
    print("\nconfidence counts:")
    print(pred_df["confidence"].value_counts().to_string())
    print(
        f"\ntokens: prompt={total_prompt_tokens:,} completion={total_completion_tokens:,}"
    )
    print(f"cost:   ${cumulative_cost:.3f} of ${COST_CEILING_USD:.2f} ceiling")
else:
    print("no predictions produced")